# Hockey Game Video

In [ ]:
import numpy as np
import hockey.hockey_env as h_env
import sys, os

sys.path.append(os.path.abspath(".."))
from common.config import RLConfig
from train import create_agent                  

In [ ]:

AGENTS = {
    #  TD3 
    "td3_1": {"type": "td3", "model": "../logs_report/2_2/td3_pink/report_td3_pink/agent_step_1000000.pth", "config": "../logs_report/2_2/td3_pink/report_td3_pink/video_config.yaml"},
    "td3_2": {"type": "td3", "model": "../logs_report/2_2/td3_pink/report_td3_pink/agent_step_1000000.pth", "config": "../logs_report/2_2/td3_pink/report_td3_pink/video_config.yaml"},
    "td3_3": {"type": "td3", "model": "../logs_report/2_2/td3_pink/report_td3_pink/agent_final.pth", "config": "../logs_report/2_2/td3_pink/report_td3_pink/video_config.yaml"},
    # SAC
    "sac_1": {"type": "sac", "model": "../logs_report/2_2/sac_basic/report_sac_basic/agent_step_1000000.pth", "config": "../logs_report/2_2/sac_basic/report_sac_basic/video_config.yaml"},
    "sac_2": {"type": "sac", "model": "../logs_report/2_2/sac_basic/report_sac_basic/agent_step_1000000.pth", "config": "l../ogs_report/2_2/sac_basic/report_sac_basic/video_config.yaml"},
    "sac_3": {"type": "sac", "model": "../logs_report/2_2/sac_basic/report_sac_basic/agent_final.pth", "config": "../logs_report/2_2/sac_basic/report_sac_basic/video_config.yaml"},
    "sac_4": {"type": "sac", "model": "../final_model/drai_team/team_final.pth", "config": "../final_model/drai_team/team_final.yaml"},

    #bots
    "weak_bot":   {"type": "basic_weak"},
    "strong_bot": {"type": "basic_strong"},
}

In [ ]:
def _load_player(key: str):
    entry = AGENTS[key]
    agent_type = entry["type"]

    if agent_type == "basic_weak":
        return h_env.BasicOpponent(weak=True)
    if agent_type == "basic_strong":
        return h_env.BasicOpponent(weak=False)

    config = RLConfig.from_yaml(entry["config"]) 
    config.agent_type = agent_type
    agent = create_agent(config)
    agent.load(entry["model"])
    return agent


def _is_basic(key: str) -> bool:
    return AGENTS[key]["type"].startswith("basic")


def _get_action(player, obs, is_basic: bool) -> np.ndarray:
    return player.act(obs) if is_basic else player.select_action(obs, eval_mode=True)

In [ ]:
def render_matchup(player1_key: str, player2_key: str = "strong_bot", num_episodes: int = 5):
    
    print(f"\n{'='*40}")
    print(f"  {player1_key}  vs  {player2_key}")
    print(f"{'='*40}\n")

    p1 = _load_player(player1_key)
    p2 = _load_player(player2_key)
    p1_basic = _is_basic(player1_key)
    p2_basic = _is_basic(player2_key)

    env = h_env.HockeyEnv(mode=h_env.Mode.NORMAL)

    for episode in range(num_episodes):
        obs, _ = env.reset()
        done = False
        total_reward = 0

        while not done:
            env.render()
            a1 = _get_action(p1, obs,                  p1_basic)
            a2 = _get_action(p2, env.obs_agent_two(),  p2_basic)
            obs, reward, terminated, truncated, info = env.step(np.hstack([a1, a2]))
            total_reward += reward
            done = terminated or truncated

        winner = info.get("winner", 0)
        winner_str = {1: player1_key, -1: player2_key, 0: "draw"}.get(winner, "?")
        print(f"Episode {episode+1:>2}: reward = {total_reward:+.2f} | winner: {winner_str}")

    env.close()

In [ ]:
render_matchup("sac_1", num_episodes=70)  

In [ ]:
render_matchup("td3_1",  num_episodes=70)  

In [ ]:
render_matchup("sac_3", num_episodes=70)  

In [ ]:
render_matchup("td3_3",  num_episodes=70)  


  td3_1  vs  strong_bot

Using device: mps
Loaded weights from timestep 0 (fine-tune mode)
Episode  1: reward = +8.57 | winner: td3_1
Episode  2: reward = +9.29 | winner: td3_1
Episode  3: reward = +7.88 | winner: td3_1
Episode  4: reward = +8.16 | winner: td3_1
Episode  5: reward = +7.04 | winner: td3_1
Episode  6: reward = +9.44 | winner: td3_1
Episode  7: reward = +9.22 | winner: td3_1
Episode  8: reward = -11.48 | winner: strong_bot
Episode  9: reward = +9.10 | winner: td3_1
Episode 10: reward = +9.06 | winner: td3_1
Episode 11: reward = +8.99 | winner: td3_1
Episode 12: reward = +9.58 | winner: td3_1
Episode 13: reward = +9.21 | winner: td3_1
Episode 14: reward = +9.30 | winner: td3_1
Episode 15: reward = +9.09 | winner: td3_1
Episode 16: reward = +9.61 | winner: td3_1
Episode 17: reward = +8.81 | winner: td3_1
Episode 18: reward = +9.51 | winner: td3_1
Episode 19: reward = +9.44 | winner: td3_1
Episode 20: reward = +8.15 | winner: td3_1
Episode 21: reward = +8.96 | winner: td3_1

KeyboardInterrupt: 

In [ ]:
render_matchup("sac_4", "sac_3", num_episodes=70)  


  sac_4  vs  sac_3

Using device: mps
Using device: mps
Episode  1: reward = +9.79 | winner: sac_4
Episode  2: reward = +8.84 | winner: sac_4
Episode  3: reward = +9.78 | winner: sac_4
Episode  4: reward = +8.36 | winner: sac_4
Episode  5: reward = +9.69 | winner: sac_4
Episode  6: reward = +9.59 | winner: sac_4
Episode  7: reward = +9.78 | winner: sac_4
Episode  8: reward = +8.71 | winner: sac_4
Episode  9: reward = +9.78 | winner: sac_4
Episode 10: reward = +8.84 | winner: sac_4
Episode 11: reward = +9.71 | winner: sac_4
Episode 12: reward = +9.25 | winner: sac_4
Episode 13: reward = +9.84 | winner: sac_4
Episode 14: reward = +9.57 | winner: sac_4
Episode 15: reward = +9.79 | winner: sac_4
Episode 16: reward = +9.60 | winner: sac_4
Episode 17: reward = +9.71 | winner: sac_4
Episode 18: reward = +8.47 | winner: sac_4
Episode 19: reward = +9.79 | winner: sac_4
Episode 20: reward = +8.48 | winner: sac_4
Episode 21: reward = +9.82 | winner: sac_4
Episode 22: reward = +9.58 | winner: sac